# log-samples-eval-callback composite — cx21: every-K-steps callback logs generated samples as wandb.Image

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `log-samples-eval-callback`, `wandb-log-step`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader, TensorDataset

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "log-samples-eval-callback"
DD_ATOM_IDS = ["log-samples-eval-callback", "wandb-log-step"]
DD_SUBTOPICS = ["Logging: log-samples eval callback", "Logging: wandb.log step"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

For GAN / VAE training, the canonical eval signal is **generated image samples**. The way you ship them to wandb is to wrap each tensor in `wandb.Image(...)` and call `wandb.log({'samples': [...], 'step': step})`.

**The two atoms.**
- **log-samples-eval-callback** — fire every `K` steps; produce `N` samples from the generator.
- **wandb-log-step** — call `wandb.log(payload)` with the step number and the wrapped samples. `wandb.Image` is `wandb`'s tagged-image type; the dashboard renders it as a panel of thumbnails.

**Anatomy.**
```python
if step % log_every == 0:
    samples = generator(z)                # shape (N, C, H, W)
    images = [wandb.Image(s) for s in samples]
    wandb.log({'samples': images, 'step': step})
```

**Why care about the wrapper.** `wandb.log({'samples': tensor})` silently logs a histogram, not an image panel. The `wandb.Image(...)` wrap is what tells wandb 'this is a picture, render it as one'. The test confirms `wandb.Image` was called exactly N times per fire.

### Composite Exercise — every-K-steps callback logs generated samples as wandb.Image

**Atoms exercised together**: `log-samples-eval-callback`, `wandb-log-step`

Implement `cx21_log_samples_to_wandb(n_iters, log_every, generator, wandb)`.

Inputs:
- `n_iters` — int.
- `log_every` — int. Fire callback when `step % log_every == 0` (and `step != 0`).
- `generator` — callable `(step: int) -> t.Tensor`. Returns a batch of shape `(N, C, H, W)`.
- `wandb` — the wandb module (mocked in tests).

Required behaviour:
1. Initialise `step = 0`.
2. For each iteration:
   - Increment `step`.
   - If `step % log_every == 0`:
     - Call `samples = generator(step)`; iterate the batch dim to get individual sample tensors.
     - Wrap each in `wandb.Image(sample)` (atom: wandb.Image is wandb's image type).
     - Call `wandb.log({'samples': images, 'step': step})` (atom: wandb-log-step).
3. Return the final step.

Test checks:
- `wandb.log.call_count == n_iters // log_every`.
- Each `wandb.log` payload has key `'samples'` (a list) and key `'step'` (the step number).
- `wandb.Image` was called once per individual sample (so `N * (n_iters // log_every)` times total).
- `wandb.Image` was called with a tensor of the per-sample shape `(C, H, W)` — not the full batch.

In [ ]:
def cx21_log_samples_to_wandb(n_iters, log_every, generator, wandb):
    step = 0
    for _ in range(n_iters):
        step += 1
        # Atom A (log-samples-eval-callback): fire every log_every steps.
        if step % log_every == 0:
            samples = generator(step)
            # Wrap each sample in wandb.Image (one wrap per sample, not per batch).
            images = [wandb.Image(s) for s in samples]
            # Atom B (wandb-log-step): log payload with samples + step.
            wandb.log({'samples': images, 'step': step})
    return step


<details><summary>Show solution — cx21</summary>

```python
def cx21_log_samples_to_wandb(n_iters, log_every, generator, wandb):
    step = 0
    for _ in range(n_iters):
        step += 1
        # Atom A (log-samples-eval-callback): fire every log_every steps.
        if step % log_every == 0:
            samples = generator(step)
            # Wrap each sample in wandb.Image (one wrap per sample, not per batch).
            images = [wandb.Image(s) for s in samples]
            # Atom B (wandb-log-step): log payload with samples + step.
            wandb.log({'samples': images, 'step': step})
    return step
```

Iterating `for s in samples` over a tensor of shape `(N, C, H, W)` yields per-sample tensors of shape `(C, H, W)` — exactly what `wandb.Image` expects. If you pass the whole batch as a single `wandb.Image(samples)`, the dashboard tries to render a 4-D array as one image and either fails or shows nonsense. The 1-wrap-per-sample contract is the test's main signal.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx21'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx21',
        'subtopics': ["Logging: log-samples eval callback", "Logging: wandb.log step"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()